### Ce notebook permet de faire une vérification de la qualité des données avant toute consolidation, visualisation ou modélisation.

In [27]:
import pandas as pd
from pathlib import Path

###  Chargement de données

In [32]:
# Chemin d'accès aux données brutes

RAW_DATA_DIR = Path("../..") / "data" / "raw"

# Charger les datasets

files = {
    "idmc": "data_idmc_depuis_2000.csv",
    "solutions": "data_solutions_depuis_2000.csv",
    "decisions": "decisions_asile_depuis_2000.csv",
    "demandes": "demandes_asile_depuis_2000.csv",
    "demographie": "demographie_depuis_2000.csv",
    "pays": "countries.csv"
}


dfs = {
    name: pd.read_csv(RAW_DATA_DIR / filename)
    for name, filename in files.items()
}

for name, df in dfs.items():
    print(f"{name:15} : {df.shape[0]:,} lignes × {df.shape[1]} colonnes")

idmc            : 934 lignes × 10 colonnes
solutions       : 21,258 lignes × 13 colonnes
decisions       : 113,929 lignes × 17 colonnes
demandes        : 120,597 lignes × 14 colonnes
demographie     : 116,781 lignes × 24 colonnes
pays            : 232 lignes × 16 colonnes


### Cohérence
On cherche maintenant les différences susceptibles de casser les futures jointures.

In [33]:
text_cols = [
    "coo",
    "coo_name",
    "coo_iso",
    "coa",
    "coa_name",
    "coa_iso"
]

for name, df in dfs.items():

    for col in text_cols:

        if col in df.columns:

            nb_spaces = (
                df[col].dropna().astype(str)
                != df[col].dropna().astype(str).str.strip()
            ).sum()

            print(name, col, "espaces :", nb_spaces)

idmc coo espaces : 0
idmc coo_name espaces : 22
idmc coo_iso espaces : 0
idmc coa espaces : 0
idmc coa_name espaces : 22
idmc coa_iso espaces : 0
solutions coo espaces : 0
solutions coo_name espaces : 597
solutions coo_iso espaces : 0
solutions coa espaces : 0
solutions coa_name espaces : 138
solutions coa_iso espaces : 0
decisions coo espaces : 0
decisions coo_name espaces : 1166
decisions coo_iso espaces : 0
decisions coa espaces : 0
decisions coa_name espaces : 20
decisions coa_iso espaces : 0
demandes coo espaces : 0
demandes coo_name espaces : 1187
demandes coo_iso espaces : 0
demandes coa espaces : 0
demandes coa_name espaces : 20
demandes coa_iso espaces : 0
demographie coo espaces : 0
demographie coo_name espaces : 1282
demographie coo_iso espaces : 0
demographie coa espaces : 0
demographie coa_name espaces : 52
demographie coa_iso espaces : 0


## Validité des années

In [34]:
for name, df in dfs.items():

    # Exclure le df "pays"
    if name == "pays":
        continue

    year = pd.to_numeric(df["year"], errors="coerce")

    invalid = df[
        year.isna() |
        ~year.between(2000, 2025)
    ]

    print(
        name,
        "min =", year.min(),
        "max =", year.max(),
        "invalides =", len(invalid)
    )

idmc min = 1990 max = 2025 invalides = 10
solutions min = 1959 max = 2025 invalides = 5443
decisions min = 2000 max = 2025 invalides = 0
demandes min = 2000 max = 2025 invalides = 0
demographie min = 2001 max = 2025 invalides = 0


### Validité des codes pays
C'est essentiel puisque coo_iso et coa_iso vont servir aux analyses géographiques et aux jointures.

In [35]:
for name, df in dfs.items():

    for col in ["coo_iso", "coa_iso"]:

        if col in df.columns:

            print(
                name,
                col,
                df[col].nunique(dropna=False)
            )

            print(
                sorted(
                    df[col]
                    .dropna()
                    .astype(str)
                    .unique()
                )[:30]
            )

idmc coo_iso 84
['AB9', 'AFG', 'ARM', 'AZE', 'BDI', 'BEN', 'BFA', 'BGD', 'BIH', 'BOL', 'BRA', 'CAF', 'CIV', 'CMR', 'COD', 'COG', 'COL', 'COM', 'CYP', 'DZA', 'ECU', 'EGY', 'ERI', 'ETH', 'GEO', 'GHA', 'GMB', 'GTM', 'HND', 'HRV']
idmc coa_iso 84
['AB9', 'AFG', 'ARM', 'AZE', 'BDI', 'BEN', 'BFA', 'BGD', 'BIH', 'BOL', 'BRA', 'CAF', 'CIV', 'CMR', 'COD', 'COG', 'COL', 'COM', 'CYP', 'DZA', 'ECU', 'EGY', 'ERI', 'ETH', 'GEO', 'GHA', 'GMB', 'GTM', 'HND', 'HRV']
solutions coo_iso 189
['AFG', 'AGO', 'ALB', 'ARE', 'ARG', 'ARM', 'ATG', 'AUS', 'AUT', 'AZE', 'BDI', 'BEL', 'BEN', 'BFA', 'BGD', 'BGR', 'BHR', 'BHS', 'BIH', 'BLR', 'BLZ', 'BOL', 'BRA', 'BRB', 'BRN', 'BTN', 'BWA', 'CAF', 'CAN', 'CHE']
solutions coa_iso 171
['AFG', 'AGO', 'ALB', 'ARE', 'ARG', 'ARM', 'ATG', 'AUS', 'AUT', 'AZE', 'BDI', 'BEL', 'BEN', 'BFA', 'BGD', 'BGR', 'BHR', 'BHS', 'BIH', 'BLR', 'BLZ', 'BOL', 'BRA', 'BWA', 'CAF', 'CAN', 'CHE', 'CHL', 'CHN', 'CIV']
decisions coo_iso 211
['AFG', 'AGO', 'ALB', 'AND', 'ARE', 'ARG', 'ARM', 'ATG', '

In [36]:
pattern_iso3 = r"^[A-Z]{3}$"

for name, df in dfs.items():

    for col in ["coo_iso", "coa_iso"]:

        if col in df.columns:

            values = df[col].astype("string").str.strip().str.upper()

            invalid = (
                values.notna()
                & ~values.str.match(pattern_iso3, na=False)
            )

            print(name, col, "formats invalides :", invalid.sum())

idmc coo_iso formats invalides : 12
idmc coa_iso formats invalides : 12
solutions coo_iso formats invalides : 0
solutions coa_iso formats invalides : 0
decisions coo_iso formats invalides : 0
decisions coa_iso formats invalides : 0
demandes coo_iso formats invalides : 0
demandes coa_iso formats invalides : 0
demographie coo_iso formats invalides : 0
demographie coa_iso formats invalides : 0


Vérification des formats invalides de IDMC

In [37]:
idmc = dfs["idmc"]

In [38]:
for col in ["coo_iso", "coa_iso"]:

    invalid_mask = (
        idmc[col].isna() |
        ~idmc[col]
            .astype(str)
            .str.strip()
            .str.upper()
            .str.fullmatch(r"[A-Z]{3}")
    )

    print(f"\n--- {col} ---")

    print(
        idmc.loc[invalid_mask, col]
            .value_counts(dropna=False)
    )


--- coo_iso ---
coo_iso
AB9    12
Name: count, dtype: int64

--- coa_iso ---
coa_iso
AB9    12
Name: count, dtype: int64


Format invalide : invalide, car le troisième caractère est un chiffre

### Validité des mesures numériques

In [39]:
numeric_expected = {
    "idmc": ["total"],

    "solutions": [
        "returned_refugees",
        "resettlement",
        "naturalisation",
        "returned_idps"
    ],

    "decisions": [
        "dec_recognized",
        "dec_other",
        "dec_rejected",
        "dec_closed",
        "dec_total"
    ],

    "demandes": ["applied"],

    "demographie": [
        "f_0_4", "f_5_11", "f_12_", "f_18_59",
        "f_60", "f_other", "f_total",
        "m_0_4", "m_5_11", "m_12_17", "m_18_59",
        "m_60", "m_other", "m_total",
        "total"
    ]
}

In [40]:
for name, cols in numeric_expected.items():

    df = dfs[name]

    for col in cols:

        if col not in df.columns:
            continue

        converted = pd.to_numeric(df[col], errors="coerce")

        conversion_errors = (
            df[col].notna() &
            converted.isna()
        ).sum()

        negative = (converted < 0).sum()

        print(
            name,
            col,
            "non numériques =", conversion_errors,
            "| négatifs =", negative
        )

idmc total non numériques = 0 | négatifs = 0
solutions returned_refugees non numériques = 15161 | négatifs = 0
solutions resettlement non numériques = 11644 | négatifs = 0
solutions naturalisation non numériques = 13872 | négatifs = 0
solutions returned_idps non numériques = 20878 | négatifs = 0
decisions dec_recognized non numériques = 0 | négatifs = 0
decisions dec_other non numériques = 0 | négatifs = 0
decisions dec_rejected non numériques = 0 | négatifs = 0
decisions dec_closed non numériques = 0 | négatifs = 0
decisions dec_total non numériques = 0 | négatifs = 0
demandes applied non numériques = 0 | négatifs = 0
demographie f_0_4 non numériques = 0 | négatifs = 0
demographie f_5_11 non numériques = 0 | négatifs = 0
demographie f_18_59 non numériques = 0 | négatifs = 0
demographie f_60 non numériques = 0 | négatifs = 0
demographie f_other non numériques = 0 | négatifs = 0
demographie f_total non numériques = 0 | négatifs = 0
demographie m_0_4 non numériques = 0 | négatifs = 0
dem

Identification des valeurs non numériques de df solutions

In [41]:
df = dfs["solutions"]

cols = [
    "returned_refugees",
    "resettlement",
    "naturalisation",
    "returned_idps"
]

for col in cols:

    converted = pd.to_numeric(df[col], errors="coerce")

    invalid_mask = (
        df[col].notna() &
        converted.isna()
    )

    print(f"\n--- {col} ---")
    print(f"Nombre d'anomalies : {invalid_mask.sum()}")

    print(
        df.loc[invalid_mask, col]
        .value_counts(dropna=False)
        .head(20)
    )


--- returned_refugees ---
Nombre d'anomalies : 15161
returned_refugees
-    15161
Name: count, dtype: int64

--- resettlement ---
Nombre d'anomalies : 11644
resettlement
-    11644
Name: count, dtype: int64

--- naturalisation ---
Nombre d'anomalies : 13872
naturalisation
-    13872
Name: count, dtype: int64

--- returned_idps ---
Nombre d'anomalies : 20878
returned_idps
-    20878
Name: count, dtype: int64
